# FT-02 : QLoRA — Fine-Tuning avec Quantization

**Objectif** : Comprendre la quantization 4-bit (NF4) et l'approche QLoRA qui combine quantization + LoRA pour fine-tuner des modeles jusqu'a 7B params sur un GPU consumer.

**Plan** :
1. Le probleme memoire des grands modeles
2. Quantization : de FP16 a NF4
3. BitsAndBytesConfig en pratique
4. QLoRA : quantized base + LoRA adapters
5. Entrainement QLoRA sur OPT-1.3B
6. Comparaison LoRA vs QLoRA
7. Exercices

**Prerequis** : FT-01 (Introduction au Fine-Tuning) — LoRA, rang, adaptateurs.

**Materiel requis** : GPU avec ~8 Go VRAM (QLoRA OPT-1.3B 4-bit = ~2 Go).

In [1]:
import torch
import os
import gc
import time

print(f"PyTorch {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM totale : {vram_total:.1f} Go | Libre : {vram_free:.1f} Go")

PyTorch 2.11.0+cu126
CUDA : True
GPU : NVIDIA GeForce RTX 3090


VRAM totale : 25.8 Go | Libre : 24.5 Go


## 1. Le probleme memoire des grands modeles

Fine-tuner un modele de langue necessite de stocker en VRAM :

| Composant | Taille approximative |
|-----------|---------------------|
| Poids du modele (FP16) | 2 bytes/param |
| Gradients | 2 bytes/param |
| Etats de l'optimiseur (AdamW) | 8 bytes/param (m + v) |
| Activations (forward pass) | Variable, souvent 1-2x modele |

**Total pour full fine-tuning** : ~12-16 bytes par parametre.

| Modele | Params | Full FT (FP16) | LoRA (FP16) | QLoRA (4-bit) |
|--------|--------|---------------|-------------|---------------|
| GPT-2 | 124M | ~2 Go | ~0.5 Go | ~0.3 Go |
| OPT-1.3B | 1.3B | ~20 Go | ~5 Go | ~2 Go |
| Llama-7B | 7B | ~112 Go | ~28 Go | ~6 Go |

LoRA reduit les gradients et l'optimiseur (seuls les adaptateurs sont entraines), mais les **poids du modele restent en FP16**.

**QLoRA** va plus loin : il **quantize les poids du modele en 4-bit** tout en gardant les calculs en precision superieure.

## 2. Quantization : de FP16 a NF4

### 2a. Principe de la quantization

La quantization reduit la precision des poids pour diminuer l'empreinte memoire :

```
FP32 (32 bits) → FP16 (16 bits) → INT8 (8 bits) → NF4 (4 bits)
4 bytes/param    2 bytes/param    1 byte/param    0.5 bytes/param
```

### 2b. Pourquoi NF4 et pas un simple INT4 ?

Les poids des reseaux de neurones suivent une distribution **approximativement normale** (centree sur 0). Un encodage lineaire INT4 gaspille des valeurs possibles :

- **INT4** : valeurs uniformement reparties sur [-8, 7] → regions denses de la distribution mal representees
- **NF4 (NormalFloat 4)** : valeurs optimisees pour une distribution normale → chaque quantile represente une fraction egale de la distribution

NF4 = le format optimal theorique pour les poids de reseaux de neurones (Dettmers et al., 2023).

### 2c. Double quantization

QLoRA applique aussi une **double quantization** : les constantes de quantization (scaling factors) sont elles-memes quantizees en FP8, economisant ~0.37 bits/param supplementaire.

## 3. BitsAndBytesConfig en pratique

La bibliotheque `bitsandbytes` integre la quantization directement dans le chargement du modele via `transformers`.

In [2]:
from transformers import BitsAndBytesConfig

# Configuration QLoRA : quantization 4-bit NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat 4 (optimal pour poids)
    bnb_4bit_compute_dtype=torch.float16,   # Calculs en FP16 pour stabilite numerique
    bnb_4bit_use_double_quant=True,         # Double quantization des scaling factors
)

print("BitsAndBytesConfig QLoRA :")
print(f"  Quantization : 4-bit NF4")
print(f"  Compute dtype : float16")
print(f"  Double quant : True")

BitsAndBytesConfig QLoRA :
  Quantization : 4-bit NF4
  Compute dtype : float16
  Double quant : True


In [3]:
# Mesurer la VRAM avant chargement
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    vram_before = torch.cuda.memory_allocated() / 1e9
else:
    vram_before = 0

print(f"VRAM avant chargement : {vram_before:.2f} Go")

VRAM avant chargement : 0.00 Go


## 4. QLoRA : quantized base + LoRA adapters

Chargeons OPT-1.3B en 4-bit et comparons avec le chargement FP16 classique.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "facebook/opt-1.3b"  # 1.3B params

# Charger en 4-bit (QLoRA)
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Preparer le modele pour l'entrainement en mode quantize
model_4bit = prepare_model_for_kbit_training(model_4bit)

# Statistiques memoire
if torch.cuda.is_available():
    vram_4bit = torch.cuda.max_memory_allocated() / 1e9
else:
    vram_4bit = 0

total_params = sum(p.numel() for p in model_4bit.parameters())
print(f"Modele : {MODEL_NAME}")
print(f"Params totaux : {total_params:,} ({total_params/1e9:.2f}B)")
print(f"VRAM apres chargement 4-bit : {vram_4bit:.2f} Go")
print(f"Theorie 4-bit : {total_params * 0.5 / 1e9:.2f} Go (poids seuls)")

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

W0524 04:06:56.638000 63876 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Modele : facebook/opt-1.3b
Params totaux : 711,778,304 (0.71B)
VRAM apres chargement 4-bit : 0.99 Go
Theorie 4-bit : 0.36 Go (poids seuls)


In [5]:
# Ajouter les adaptateurs LoRA sur le modele quantize
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Attention query/value dans OPT
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_qlora = get_peft_model(model_4bit, lora_config)
model_qlora.print_trainable_parameters()

trainable = sum(p.numel() for p in model_qlora.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_qlora.parameters())
print(f"\nParams entrainables : {trainable:,} ({trainable/total*100:.2f}%)")

trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.2385

Params entrainables : 3,145,728 (0.44%)


In [6]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer : vocabulaire = {tokenizer.vocab_size:,} tokens")

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Tokenizer : vocabulaire = 50,265 tokens


## 5. Entrainement QLoRA sur OPT-1.3B

Fine-tunons OPT-1.3B sur du texte francais litteraire, le meme type de dataset que FT-01.

In [7]:
from datasets import Dataset

# Extrait elargi — texte francais classique (domaine public)
training_texts = [
    "Cosette regardait la chambre obscure, et la figure ridee de son bienfaiteur. "
    "Cette figure ridee etait l'immense figure de la misere humaine. Jean Valjean "
    "avait pris l'habitude de descendre a la cave pour rapporter du vin.",

    "La nuit, quand tout dormait dans la maison, Jean Valjean restait eveille. "
    "Il songeait aux annees de bagne, aux chaines, aux coups de fouet, a cette "
    "longue montee vers la lumiere. Cosette dormait, et il la regardait dormir.",

    "Les rues de Paris etaient sombres en ce temps-la. Les reverberes jetaient "
    "des taches de lumiere jaune sur les paves mouilles. Jean Valjean marchait "
    "vite, comme un homme qui fuit quelque chose.",

    "Marius l'observait de loin. Ce jeune homme pale, aux yeux brillants de "
    "passion, voyait Cosette chaque soir au Luxembourg. Il n'osait lui parler. "
    "L'amour est timide, meme pour les coeurs les plus courageux.",

    "La barricade s'elevait dans la rue. Les jeunes hommes avaient pris les "
    "paves et construit un mur de pierres. Enjolras commandait avec une "
    "autorite naturelle. Grantaire, ivre et fidele, le suivait partout.",

    "Gavroche passait entre les balles comme un oiseau. Ce gamin de Paris, "
    "nenfant abandonne devenu enfant de la patrie, ramassait les cartouches "
    "des soldats morts pour les rapporter aux insurges.",

    "Eponine errait dans les rues, le coeur lourd d'un amour sans espoir. "
    "Elle avait vu Marius regarder Cosette, et elle avait compris que son "
    "propre amour ne serait jamais partage.",

    "Javet suivait la piste avec une tenacite implacable. Cet homme de loi "
    "ne connaissait ni pitié ni doute. Pour lui, Jean Valjean resterait "
    "toujours le forcat 24601, quelles que soient ses actions.",

    "Le jardin du Luxembourg etait paisible en ce matin de printemps. "
    "Les marronniers etaient en fleur, et les enfants jouaient sur le sable. "
    "Cosette lisait sur un banc, attendant celui qui ne viendrait pas.",

    "Les Thenardier comptaient leur butin dans l'arriere-salle de l'auberge. "
    "Monsieur Thenardier calculait, madame Thenardier injuriait. La cupidite "
    "etait leur religion, l'argent leur seul dieu.",
]

train_data = Dataset.from_dict({"text": training_texts})
print(f"Dataset : {len(train_data)} exemples")
print(f"Moyenne caracteres/exemple : {sum(len(t) for t in training_texts) / len(training_texts):.0f}")

Dataset : 10 exemples
Moyenne caracteres/exemple : 200


In [8]:
# Tokenisation
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Tokens par exemple : {len(tokenized_dataset[0]['input_ids'])}")
print(f"Dataset tokenise : {len(tokenized_dataset)} exemples")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokens par exemple : 256
Dataset tokenise : 10 exemples


In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Arguments d'entrainement QLoRA
training_args = TrainingArguments(
    output_dir="./temp_ft02_output",
    num_train_epochs=15,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
    optim="paged_adamw_8bit",  # Optimiseur 8-bit pour economiser la VRAM
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model_qlora,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Mesurer VRAM avant/apres
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    vram_start = torch.cuda.memory_allocated() / 1e9
else:
    vram_start = 0

start_time = time.time()
print("Entrainement QLoRA en cours...")
train_result = trainer.train()
elapsed = time.time() - start_time

if torch.cuda.is_available():
    vram_peak = torch.cuda.max_memory_allocated() / 1e9
else:
    vram_peak = 0

print(f"\nPerte finale : {train_result.training_loss:.4f}")
print(f"Temps : {elapsed:.1f}s")
print(f"VRAM pic entrainement : {vram_peak:.2f} Go")

Entrainement QLoRA en cours...


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\autograd\graph.py:869: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\au

Step,Training Loss
10,3.378780
20,2.853671
30,2.530290
40,2.292382
50,2.111398
60,2.000296
70,1.929940



Perte finale : 2.4067
Temps : 69.5s
VRAM pic entrainement : 1.16 Go


## 6. Generation avec le modele QLoRA

Comparons la generation avant et apres fine-tuning.

In [10]:
# Generer avec le modele fine-tune
model_qlora.eval()

def generate_qlora(prompt, max_new_tokens=100, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt").to(model_qlora.device)
    with torch.no_grad():
        outputs = model_qlora.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompts = [
    "Dans les rues sombres de Paris,",
    "Jean Valjean regardait la nuit",
    "Cosette marchait dans le jardin",
]

for p in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt : \"{p}\"")
    print(f"{'='*60}")
    output = generate_qlora(p)
    print(output)


Prompt : "Dans les rues sombres de Paris,"


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Dans les rues sombres de Paris, Jean Valjean lui restait la sonne du consaveau. L'homme de la paix ne lui avait pas fait ce qu'il lui aveuglait. Jean Valjean ne porterait quelque chose quelque temps. Il n'avait jamais vu Cosette, la bienveille de la patrie, s'entendre comme une enfant pour la poursu

Prompt : "Jean Valjean regardait la nuit"


Jean Valjean regardait la nuit dans un bar en Valette. Il n'en avait jamais vu ce matin de soirée. Cosette lui avait donné le gaz pour la nuit, et il lui avait dit qu'il serait le reste de l'eau. Le huitième Parisien, qui n'avait jamais vu la soirée comme une joue de football, aurait

Prompt : "Cosette marchait dans le jardin"


Cosette marchait dans le jardin, comme un jeune homme dans une rue où les gendarmes ne peaient jamais dormir. Les coeurs etaient la force de l'amour, et Cosette marchait pour lui. Elle ne pouvait pas marcher pour Jean Valjean.
Cette soirée, Cosette marchait dans les rues, comme un homme dans une rue o�


## 7. Comparaison LoRA vs QLoRA

Chargeons le meme modele en FP16 (sans quantization) pour comparer l'empreinte memoire.

In [11]:
# Charger le modele en FP16 pour comparaison
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Ajouter LoRA identique
lora_config_fp16 = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model_lora_fp16 = get_peft_model(model_fp16, lora_config_fp16)

# Mesures comparatives
def measure_vram(model, label):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"{label} : alloue={alloc:.2f} Go, reserve={reserved:.2f} Go")
        return alloc
    return 0

print(f"\n{'='*50}")
print(f"Comparaison memoire — {MODEL_NAME}")
print(f"{'='*50}")

# Nettoyer pour mesure propre
del model_fp16, model_lora_fp16
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Recharger pour mesure propre
torch.cuda.reset_peak_memory_stats()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)
vram_fp16 = torch.cuda.max_memory_allocated() / 1e9

del model_fp16
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_4b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
vram_4bit_load = torch.cuda.max_memory_allocated() / 1e9

del model_4b
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n{'Configuration':<25} {'VRAM':>10} {'Economie':>10}")
print(f"{'-'*25} {'-'*10} {'-'*10}")
print(f"{'FP16 (base)':<25} {vram_fp16:>9.2f} G {'ref':>10}")
print(f"{'4-bit NF4 (QLoRA base)':<25} {vram_4bit_load:>9.2f} G {(1-vram_4bit_load/vram_fp16)*100:>9.1f}%")
print(f"{'QLoRA (4-bit + LoRA r=16)':<25} {vram_peak:>9.2f} G {(1-vram_peak/vram_fp16)*100:>9.1f}%")
print(f"\nTheorie : FP16 = {total_params*2/1e9:.2f} Go vs 4-bit = {total_params*0.5/1e9:.2f} Go (ratio 4x)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


Comparaison memoire — facebook/opt-1.3b


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Configuration                   VRAM   Economie
------------------------- ---------- ----------
FP16 (base)                    2.55 G        ref
4-bit NF4 (QLoRA base)         1.60 G      37.3%
QLoRA (4-bit + LoRA r=16)      1.16 G      54.4%

Theorie : FP16 = 1.42 Go vs 4-bit = 0.36 Go (ratio 4x)


In [12]:
# Tableau recapitulatif complet
print("\n" + "=" * 70)
print("RESUME COMPARATIF : LoRA (FP16) vs QLoRA (4-bit)")
print("=" * 70)
print(f"\n{'Aspect':<30} {'LoRA (FP16)':<20} {'QLoRA (4-bit)':<20}")
print(f"{'-'*30} {'-'*20} {'-'*20}")
print(f"{'Precision poids':<30} {'FP16 (16 bits)':<20} {'NF4 (4 bits)':<20}")
print(f"{'Calculs forward':<30} {'FP16':<20} {'FP16 (dequant)':<20}")
print(f"{'Grad + optimizer':<30} {'FP16 (LoRA seul)':<20} {'FP16 (LoRA seul)':<20}")
print(f"{'VRAM modele ~1.3B':<30} {'~2.6 Go':<20} {'~0.7 Go':<20}")
print(f"{'Qualite vs full FT':<30} {'~98-99%':<20} {'~97-99%':<20}")
print(f"{'Vitesse entrainement':<30} {'1x (ref)':<20} {'~0.8x (dequant)':<20}")
print(f"\nRecommendation :\n"
      "  - Modele < 1B + VRAM suffisante -> LoRA (FP16) : plus simple, plus rapide\n"
      "  - Modele > 1B ou VRAM limitee -> QLoRA : Economie memoire 3-4x")


RESUME COMPARATIF : LoRA (FP16) vs QLoRA (4-bit)

Aspect                         LoRA (FP16)          QLoRA (4-bit)       
------------------------------ -------------------- --------------------
Precision poids                FP16 (16 bits)       NF4 (4 bits)        
Calculs forward                FP16                 FP16 (dequant)      
Grad + optimizer               FP16 (LoRA seul)     FP16 (LoRA seul)    
VRAM modele ~1.3B              ~2.6 Go              ~0.7 Go             
Qualite vs full FT             ~98-99%              ~97-99%             
Vitesse entrainement           1x (ref)             ~0.8x (dequant)     

Recommendation :
  - Modele < 1B + VRAM suffisante -> LoRA (FP16) : plus simple, plus rapide
  - Modele > 1B ou VRAM limitee -> QLoRA : Economie memoire 3-4x


## 8. Nettoyage memoire GPU

In [13]:
del model_qlora, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} Go")
else:
    print("Nettoyage termine")

VRAM libre : 23.5 Go


## 9. Exercices

Appliquez les concepts de ce notebook.

In [14]:
# Exercice 1 : Impact de la precision
# TODO etudiant : Comparez INT8 (load_in_8bit=True) vs NF4 vs FP16
# sur le meme modele. Mesurez la VRAM et la qualite de generation.
# Indice : creez 3 BitsAndBytesConfig differents et chargez le modele.

print("Exercice a completer : comparez INT8 vs NF4 vs FP16")

Exercice a completer : comparez INT8 vs NF4 vs FP16


In [15]:
# Exercice 2 : LoRA rank et memoire
# TODO etudiant : Testez r=4, r=16, r=64 avec QLoRA sur OPT-1.3B.
# Pour chaque rang, notez : params entrainables, VRAM pic, perte finale.
# Etape 1 : definissez les 3 LoraConfig
# Etape 2 : lancez l'entrainement pour chaque
# Etape 3 : comparez dans un tableau

print("Exercice a completer : comparez les rangs QLoRA r=4, 16, 64")

Exercice a completer : comparez les rangs QLoRA r=4, 16, 64


In [16]:
# Exercice 3 : Double quantization on/off
# TODO etudiant : Comparez bnb_4bit_use_double_quant=True vs False.
# Mesurez la difference de VRAM. Est-ce que la qualite change ?
# Indice : la double quantization economise ~0.37 bits/param.

print("Exercice a completer : comparez double quantization on/off")

Exercice a completer : comparez double quantization on/off


## Resume

| Aspect | LoRA (FT-01) | QLoRA (FT-02) |
|--------|-------------|---------------|
| Poids du modele | FP16 (16 bits) | NF4 (4 bits) |
| Adaptateurs | LoRA en FP16 | LoRA en FP16 |
| Calculs forward | FP16 natif | Dequantize → FP16 → re-quantize |
| Economie memoire | ~1.1x modele base | **~3-4x vs FP16** |
| Qualite | ~98-99% du full FT | ~97-99% du full FT |
| Modele recommande | < 1B params | **> 1B params** |

**Points cles** :
- **QLoRA = Quantization (NF4) + LoRA** : le meilleur des deux mondes
- **NF4** est optimal pour les poids (distribution normale), pas un simple INT4
- **Double quantization** compresse les scaling factors pour des gains supplementaires
- **`paged_adamw_8bit`** : optimiseur en 8-bit pour economiser encore plus de VRAM
- Un modele 7B en QLoRA tient sur un GPU 6 Go — democratise le fine-tuning

**Reference** : Dettmers et al., "QLoRA: Efficient Finetuning of Quantized LLMs" (2023).

**Prochaines etapes** (FT-03) : Fine-tuning multi-taches avec plusieurs adaptateurs LoRA.